# Surface3D — inline 3-D surfaces (PoC)

Rotatable WebGL surfaces (echarts-gl), themed from signum's `THEMES`, rendered **inline** in the notebook — same display API as `Chart` / `StatChart` (`.show()` / last-expression `_repr_html_`).

Drag to rotate · scroll to zoom · hover for labelled x / y / z. echarts is vendored + inlined, so the cells work offline.

> **Run All** (the first cell loads the in-repo `src/`). If an inline surface ever shows blank in VSCode, run `python surface_demo.py` and open `surface_iv.html` / `surface_bs.html` in a browser.

In [ ]:
import sys, math
from pathlib import Path
import numpy as np

# Locate THIS branch's src/ (the one with Surface3D + vendored echarts),
# regardless of how the notebook is opened (VSCode workspace may be a sibling).
_starts = []
_nb = globals().get("__vsc_ipynb_file__") or globals().get("__file__")
if _nb:
    _starts.append(Path(_nb).resolve().parent)
_starts.append(Path.cwd())

_src = None
for _s in _starts:
    for _d in [_s, *_s.parents]:
        if (_d / "src" / "signum" / "engine" / "surface3d.py").exists():
            _src = _d / "src"
            break
    if _src:
        break

if _src:                                   # prefer in-repo src over any pip-installed signum
    sys.path.insert(0, str(_src))
    for _m in [m for m in sys.modules if m == "signum" or m.startswith("signum.")]:
        del sys.modules[_m]

from signum.engine.surface3d import Surface3D
print("Surface3D ready:", Surface3D.__module__, "\nsrc:", _src)

## Equity implied-vol surface
Smile + negative skew + decaying term structure — renders inline below.

In [ ]:
ttm       = np.linspace(0.05, 2.0, 28)      # x: time to maturity (yrs)
moneyness = np.linspace(0.7, 1.3, 28)       # y: K / S
TT, KK = np.meshgrid(ttm, moneyness)        # (ny, nx)
iv = (0.18 + 0.05*np.exp(-1.6*TT)) - 0.07*(KK - 1.0) + 0.30*(KK - 1.0)**2/np.sqrt(TT + 0.10)

Surface3D(theme="midnight", height=560, title="Synthetic equity IV surface",
          colorscale="viridis").surface(
    ttm, moneyness, iv,
    x_label="TTM (yrs)", y_label="Moneyness K/S", z_label="Implied vol",
)

## Black-Scholes call-price surface
Call value vs spot × time-to-expiry (`auto_rotate=True`, `glass` theme, `turbo` colorscale).

In [ ]:
_ncdf = np.vectorize(lambda x: 0.5*(1.0 + math.erf(x/math.sqrt(2.0))))
spot = np.linspace(60, 140, 32)             # x: spot
tau  = np.linspace(0.02, 1.0, 32)           # y: time to expiry (yrs)
K, r, sig = 100.0, 0.02, 0.25
SS, TT = np.meshgrid(spot, tau)
d1 = (np.log(SS/K) + (r + 0.5*sig**2)*TT)/(sig*np.sqrt(TT))
d2 = d1 - sig*np.sqrt(TT)
price = SS*_ncdf(d1) - K*np.exp(-r*TT)*_ncdf(d2)

Surface3D(theme="glass", height=560, title="Black-Scholes call price",
          colorscale="turbo", auto_rotate=True).surface(
    spot, tau, price,
    x_label="Spot", y_label="TTE (yrs)", z_label="Call price",
)

## DataFrame input + `.show()`
`z` can be a DataFrame (index → y, columns → x). `.show()` displays explicitly (same as returning it).

In [ ]:
import pandas as pd
x = np.linspace(-3, 3, 40)
y = np.linspace(-3, 3, 40)
XX, YY = np.meshgrid(x, y)
Z = np.sin(np.sqrt(XX**2 + YY**2)) / (np.sqrt(XX**2 + YY**2) + 0.4)   # ripple
zdf = pd.DataFrame(Z, index=np.round(y, 3), columns=np.round(x, 3))

Surface3D(theme="dark", height=520, title="DataFrame surface (sombrero)",
          colorscale="rdylbu").surface(
    z=zdf, x_label="x", y_label="y", z_label="f(x, y)",
).show()